In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.linalg import solve_banded
from IPython.display import HTML

# --- parameters ---
N     = 1000          # grid points
L     = 1.0           # box length
dx    = L / N
x     = np.linspace(0, L, N)
dt    = 1e-5
hbar  = 1.0
m     = 1.0

# --- potential barrier ---
barrier_center = 0.55 * L
barrier_width  = 0.02 * L
V0             = 1.5e5        # barrier height
V = np.zeros(N)
V[np.abs(x - barrier_center) < barrier_width / 2] = V0

# --- initial wavepacket ---
x0    = 0.25 * L
k0    = 500.0                 # packet momentum; E ~ k0^2/2m
sigma = 0.03 * L
psi   = np.exp(-(x - x0)**2 / (2*sigma**2)) * np.exp(1j * k0 * x)
psi  /= np.sqrt(np.sum(np.abs(psi)**2) * dx)

# --- build Crank-Nicolson matrices ---
# H diagonal: kinetic + potential
# kinetic uses second-order finite difference: -hbar^2/2m * (psi[i+1]-2psi[i]+psi[i-1])/dx^2
r   = 1j * dt / (2 * hbar)
diag_ke   = hbar**2 / (m * dx**2)               # kinetic energy diagonal coefficient
main_diag = 1 + r * (diag_ke + V)               # A matrix diagonal
off_diag  = -r * hbar**2 / (2 * m * dx**2) * np.ones(N - 1)  # off-diagonal

# rhs matrix diagonals (same structure, opposite sign on r)
main_diag_rhs = 1 - r * (diag_ke + V)
off_diag_rhs = -off_diag  # critical: opposite sign from A off-diagonal for CN stability

def apply_tridiag(main, off, v):
    """Multiply tridiagonal matrix (main diag + uniform off-diag) by vector v."""
    result = main * v
    result[:-1] += off * v[1:]
    result[1:]  += off * v[:-1]
    return result

def solve_tridiag(main, off, rhs):
    """Solve tridiagonal system using scipy banded solver."""
    ab = np.zeros((3, N), dtype=complex)
    ab[0, 1:]  = off           # superdiagonal
    ab[1, :]   = main          # main diagonal
    ab[2, :-1] = off           # subdiagonal
    return solve_banded((1, 1), ab, rhs)

# --- figure setup ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7),
                                gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0d0d0d')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d0d0d')
    ax.tick_params(colors='#aaaaaa')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

# main plot
prob_line, = ax1.plot(x, np.abs(psi)**2, color='#7F77DD', lw=1.5, label='|ψ|²')
re_line,   = ax1.plot(x, np.real(psi) * 0.3, color='#5DCAA5', lw=0.8,
                       alpha=0.5, label='Re(ψ) scaled')
barrier_fill = ax1.axvspan(barrier_center - barrier_width/2,
                            barrier_center + barrier_width/2,
                            color='#D85A30', alpha=0.4, label='barrier')
ax1.set_xlim(0, L)
ax1.set_ylim(-5, 60)
ax1.set_ylabel('probability density', color='#aaaaaa')
ax1.legend(loc='upper right', facecolor='#1a1a1a',
           edgecolor='#333333', labelcolor='#cccccc', fontsize=9)

# probability bars
bar_colors = ['#7F77DD', '#D85A30']
bars = ax2.bar(['transmitted', 'reflected'], [0, 0],
               color=bar_colors, width=0.4)
ax2.set_ylim(0, 1)
ax2.set_ylabel('probability', color='#aaaaaa')
ax2.axhline(1.0, color='#444444', lw=0.5, linestyle='--')

title = ax1.set_title('', color='#cccccc', fontsize=10)

steps_per_frame = 30

def update(frame):
    global psi
    for _ in range(steps_per_frame):
        rhs = apply_tridiag(main_diag_rhs, off_diag_rhs, psi)
        psi = solve_tridiag(main_diag, off_diag, rhs)
        psi[0] = psi[-1] = 0          # hard wall boundaries

        # Safety guard in case the kernel state had previously diverged.
        if not np.isfinite(psi).all():
            psi[:] = 0.0
            psi[np.argmin(np.abs(x - x0))] = 1.0 + 0j
            break

    prob = np.abs(psi)**2

    # split probability at barrier
    split = np.searchsorted(x, barrier_center)
    p_left  = np.sum(prob[:split]) * dx    # reflected
    p_right = np.sum(prob[split:]) * dx    # transmitted
    p_total = p_left + p_right

    prob_line.set_ydata(prob)
    re_line.set_ydata(np.real(psi) * 0.3)

    bars[0].set_height(p_right)            # transmitted
    bars[1].set_height(p_left)             # reflected

    t_ratio = (p_right / p_total) if p_total > 1e-15 else 0.0
    r_ratio = (p_left / p_total) if p_total > 1e-15 else 0.0

    title.set_text(
        f'T = {t_ratio:.3f}   R = {r_ratio:.3f}   '
        f'T+R = {p_total:.3f}'
    )
    return prob_line, re_line, bars[0], bars[1], title

ani = animation.FuncAnimation(fig, update, frames=200,
                               interval=30, blit=True)
plt.tight_layout()
plt.close(fig)
HTML(ani.to_jshtml())

/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:98: RuntimeWarning: overflow encountered in square
  prob = np.abs(psi)**2
/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:113: RuntimeWarning: invalid value encountered in double_scalars
  f'T = {p_right/p_total:.3f}   R = {p_left/p_total:.3f}   '
/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:43: RuntimeWarning: overflow encountered in multiply
  result = main * v
/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:44: RuntimeWarning: invalid value encountered in multiply
  result[:-1] += off * v[1:]
/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:44: RuntimeWarning: overflow encountered in add
  result[:-1] += off * v[1:]
/var/folders/9y/klhcsx6x6xdcywbp9lk_9wjh0000gn/T/ipykernel_76578/1403792144.py:45: RuntimeWarning: invalid value encountered in multiply
  result[1:]  += off * v[:-1]


ValueError: array must not contain infs or NaNs